# Lab 3 — EEG 脑电信号隐藏特征分析

**目标：** 研究 EEG 信号的隐藏特征  
**关键技能：** 频谱密度计算（Spectrogram）& 小波变换（Scalogram）

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from scipy.signal import butter, sosfiltfilt, spectrogram
import pywt

BASE = Path.cwd()
DATA = BASE / "data"
EDF = DATA / "eeg14.edf"
CSV_A = DATA / "annotations_2017_A.csv"
CSV_B = DATA / "annotations_2017_B.csv"
CSV_C = DATA / "annotations_2017_C.csv"
OUT = BASE / "output"
OUT.mkdir(exist_ok=True)

CTX = 15.0   # seizure 前后秒数
CUT = 60.0   # 低通截止频率 Hz
ORD = 6      # 滤波器阶数

## 1. 标注加载 & 发作区间提取

In [2]:
def load_labels(paths, col):
    """多专家 Majority Vote"""
    all_l = []
    for p in paths:
        if not p.exists():
            continue
        df = pd.read_csv(p, header=None)
        df.columns = [f"eeg{i}" for i in range(1, df.shape[1] + 1)]
        if col not in df.columns:
            continue
        v = pd.to_numeric(df[col], errors="coerce").fillna(0).values
        v = np.where(v > 0, 1, 0).astype(int)[1:]
        all_l.append(v)
    if not all_l:
        raise RuntimeError("无有效标注")
    L = min(len(l) for l in all_l)
    stk = np.vstack([l[:L] for l in all_l])
    mv = (stk.sum(axis=0) > len(all_l) / 2).astype(int)
    print(f"[标注] 发作秒数: {mv.sum()}/{L}")
    return mv


def seizure_segments(y):
    d = np.diff(np.r_[0, y, 0])
    s = np.where(d == 1)[0]
    e = np.where(d == -1)[0]
    return list(zip(s, e))

## 2. EEG 信号处理

In [3]:
def load_eeg(path):
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    pk = mne.pick_types(raw.info, eeg=True, exclude="bads")
    raw.pick(pk)
    return raw.get_data(), raw.times, raw.info["sfreq"], raw.ch_names


def avg_chan(data):
    return np.mean(data, axis=0)


def lowpass(x, fs):
    ny = 0.5 * fs
    sos = butter(ORD, min(CUT, ny * 0.99) / ny, btype="low", output="sos")
    return sosfiltfilt(sos, x)


def crop(sig, t, lo, hi):
    m = (t >= lo) & (t <= hi)
    return sig[..., m], t[m]

## 3. 绘图函数

In [4]:
def fig_multichannel(d_uV, t, ss, se, cn, fname):
    lo, hi = max(t[0], ss - CTX), min(t[-1], se + CTX)
    seg, tx = crop(d_uV, t, lo, hi)
    n = seg.shape[0]
    dy = np.nanpercentile(np.abs(seg), 95) * 2.5 or 100.0
    plt.figure(figsize=(14, 10))
    for i in range(n):
        plt.plot(tx, seg[i] + i * dy, lw=0.8)
    plt.axvline(ss, ls="--", lw=1.5, color="red", label="Seizure start")
    plt.axvline(se, ls="--", lw=1.5, color="orange", label="Seizure end")
    plt.yticks([i * dy for i in range(n)], cn)
    plt.xlabel("Time (s)"); plt.ylabel("Channels")
    plt.title("EEG recordings around seizure (multichannel)")
    plt.grid(alpha=0.25); plt.legend()
    plt.tight_layout()
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  -> {fname.name}")

In [5]:
def fig_avg(av, t, ss, se, fname):
    lo, hi = max(t[0], ss - CTX), min(t[-1], se + CTX)
    seg, tx = crop(av, t, lo, hi)
    plt.figure(figsize=(14, 4.5))
    plt.plot(tx, seg, lw=1.0, color="steelblue")
    plt.axvline(ss, ls="--", lw=1.5, color="red", label="Seizure start")
    plt.axvline(se, ls="--", lw=1.5, color="orange", label="Seizure end")
    plt.xlabel("Time (s)"); plt.ylabel("Amplitude (μV)")
    plt.title("Averaged EEG signal around seizure")
    plt.grid(alpha=0.3); plt.legend()
    plt.tight_layout()
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  -> {fname.name}")

In [6]:
def fig_spec(af, t, fs, ss, se, fname):
    lo, hi = max(t[0], ss - CTX), min(t[-1], se + CTX)
    seg, tx = crop(af, t, lo, hi)
    nfft = min(512, len(seg))
    no = min(384, max(0, nfft // 2))
    f, ts, S = spectrogram(seg, fs=fs, window="hann", nperseg=nfft,
                           noverlap=no, scaling="density", mode="psd")
    SdB = 10 * np.log10(S + 1e-12)
    vmin, vmax = np.percentile(SdB, 5), np.percentile(SdB, 99)
    plt.figure(figsize=(14, 5))
    plt.pcolormesh(ts + lo, f, SdB, shading="gouraud", cmap="jet",
                   vmin=vmin, vmax=vmax)
    plt.axvline(ss, ls="--", lw=1.5, color="white", label="Seizure start")
    plt.axvline(se, ls="--", lw=1.5, color="cyan", label="Seizure end")
    plt.ylim(0, 60)
    plt.xlabel("Time (s)"); plt.ylabel("Frequency (Hz)")
    plt.title("Spectrogram of low-pass filtered average EEG")
    plt.colorbar(label="PSD (dB/Hz)"); plt.legend()
    plt.tight_layout()
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  -> {fname.name}")

In [7]:
def fig_wavelet(af, t, fs, ss, se, fname):
    lo, hi = max(t[0], ss - CTX), min(t[-1], se + CTX)
    seg, tx = crop(af, t, lo, hi)
    fr = np.linspace(1, 60, 120)
    cf = pywt.central_frequency("morl")
    sc = cf * fs / fr
    co, _ = pywt.cwt(seg, sc, "morl", sampling_period=1 / fs)
    pw = 10 * np.log10(np.abs(co) ** 2 + 1e-12)
    vmin, vmax = np.percentile(pw, 5), np.percentile(pw, 99)
    plt.figure(figsize=(14, 5))
    plt.pcolormesh(tx, fr, pw, shading="gouraud", cmap="jet",
                   vmin=vmin, vmax=vmax)
    plt.axvline(ss, ls="--", lw=1.5, color="white", label="Seizure start")
    plt.axvline(se, ls="--", lw=1.5, color="cyan", label="Seizure end")
    plt.ylim(1, 60)
    plt.xlabel("Time (s)"); plt.ylabel("Frequency (Hz)")
    plt.title("Wavelet scalogram of low-pass filtered average EEG (Morlet)")
    plt.colorbar(label="Power (dB)"); plt.legend()
    plt.tight_layout()
    plt.savefig(fname, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  -> {fname.name}")

## 4. 主流程

In [8]:
print("=" * 50)
print("  Lab 3 - EEG 隐藏特征分析")
print("=" * 50)

col = EDF.stem
print(f"\n[1] 标注 Majority Vote (列: {col})")
y = load_labels([CSV_A, CSV_B, CSV_C], col)
segs = seizure_segments(y)
print(f"  发作区间: {len(segs)} 个")
for a, b in segs:
    print(f"    {a}s ~ {b}s ({b - a}s)")
ss, se = max(segs, key=lambda x: x[1] - x[0])
print(f"  选用: {ss}s ~ {se}s (持续 {se - ss}s)")

print("\n[2] 加载 EEG")
d, t, fs, cn = load_eeg(EDF)
duV = d * 1e6
print(f"  通道: {len(cn)}, 采样率: {fs} Hz, 时长: {t[-1]:.0f}s")

print("\n[3] 通道平均化")
av = avg_chan(duV)

print(f"\n[4] 低通滤波 {CUT} Hz")
af = lowpass(av, fs)

print("\n[5] 绘图")
fig_multichannel(duV, t, ss, se, cn, OUT / "01_multichannel.png")
fig_avg(av, t, ss, se, OUT / "02_average.png")
fig_spec(af, t, fs, ss, se, OUT / "03_spectrogram.png")
fig_wavelet(af, t, fs, ss, se, OUT / "04_scalogram.png")

print("\n完成.")

  Lab 3 - EEG 隐藏特征分析

[1] 标注 Majority Vote (列: eeg14)
[标注] 发作秒数: 2280/15416
  发作区间: 36 个
    254s ~ 279s (25s)
    290s ~ 627s (337s)
    647s ~ 695s (48s)
    714s ~ 739s (25s)
    760s ~ 778s (18s)
    798s ~ 814s (16s)
    831s ~ 846s (15s)
    859s ~ 972s (113s)
    991s ~ 1010s (19s)
    1031s ~ 1052s (21s)
    1083s ~ 1101s (18s)
    1121s ~ 1139s (18s)
    1142s ~ 1520s (378s)
    1530s ~ 1666s (136s)
    1705s ~ 1721s (16s)
    1739s ~ 1848s (109s)
    1873s ~ 1902s (29s)
    1907s ~ 1936s (29s)
    1944s ~ 2407s (463s)
    2413s ~ 2465s (52s)
    2477s ~ 2501s (24s)
    2508s ~ 2518s (10s)
    2526s ~ 2592s (66s)
    2605s ~ 2621s (16s)
    2632s ~ 2652s (20s)
    2668s ~ 2686s (18s)
    2708s ~ 2722s (14s)
    2739s ~ 2767s (28s)
    2781s ~ 2794s (13s)
    2831s ~ 2843s (12s)
    2871s ~ 2884s (13s)
    2966s ~ 3011s (45s)
    3289s ~ 3302s (13s)
    3329s ~ 3342s (13s)
    3506s ~ 3526s (20s)
    3533s ~ 3603s (70s)
  选用: 1944s ~ 2407s (持续 463s)

[2] 加载 EEG


C:\Users\11201\AppData\Local\Temp\ipykernel_41684\167695719.py:2: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
C:\Users\11201\AppData\Local\Temp\ipykernel_41684\167695719.py:2: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


  通道: 21, 采样率: 256.0 Hz, 时长: 3726s

[3] 通道平均化

[4] 低通滤波 60.0 Hz

[5] 绘图


C:\Users\11201\AppData\Local\Temp\ipykernel_41684\2511961717.py:15: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()


  -> 01_multichannel.png
  -> 02_average.png
  -> 03_spectrogram.png


C:\Users\11201\AppData\Local\Temp\ipykernel_41684\2792015569.py:19: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()


  -> 04_scalogram.png

完成.
